# Evaluation — Granite Speech

In [ ]:
!pip install -U -q "transformers>=4.52" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec

## Methods for Evaluating ASR

## Setup

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

MODEL_ID = "ibm-granite/granite-speech-5.0-2b"
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID, device_map=device,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
).eval()

## Kaldi Test Data

In [3]:
from pathlib import Path
import torchaudio
from IPython.display import Audio, display


dataset = []

dataset_path = Path("/training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test") #other languages *_de *_fr *_es
# other datasets
# Mozilla Commonvoice: /training-1/asr/kaldi_data/cv19/eb/test *_de *_fr *_es
# Mozilla Commonvoice Single Words: /training-1/asr/kaldi_data/singleword/de/test *_de *_fr *_es 
# Multilingual Librispeech: /training-1/asr/kaldi_data/mls/en/test *_de *_fr *_es

wav_scp_dict = {}
with open(f"{dataset_path}/wav.scp", encoding="UTF-8") as rf:
    for line in rf:
        id, wav_path = line.strip().split("\t")
        wav_scp_dict[id] = wav_path

text_dict = {}
with open(f"{dataset_path}/text", encoding="UTF-8") as rf:
    for line in rf:
        id, text = line.strip().split("\t")
        text_dict[id] = text

for id, wav_path in wav_scp_dict.items():
    text = text_dict[id]
    dataset.append({"audio" : wav_path, "text" : text})

first_entry = dataset[0]
print(first_entry)
print("-"*10)
print(first_entry["text"])
audio = torchaudio.load(first_entry["audio"])[0]
display(Audio(audio, rate=16000))

{'audio': '/training-1/asr/voxpopuli/out/transcribed_data/en/2013/20131022-0900-PLENARY-3-en_20131022-08:32:08_10.wav', 'text': 'then we as a parliament could take our responsibility and quickly vote through these measures because they will need to take effect on one january'}
----------
then we as a parliament could take our responsibility and quickly vote through these measures because they will need to take effect on one january


## Transcription Function

In [4]:
def transcribe(wav, prompt="can you transcribe the speech into a written format?", hotwords=None):
    if hotwords:
        prompt = f"{prompt}\nhotwords: {hotwords}"
    chat = [
        {"role": "system", "content": "You are Granite, developed by IBM. You are a helpful AI assistant."},
        {"role": "user", "content": f"<|audio|>{prompt}"},
    ]
    text = processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = processor(text, wav, device=device, return_tensors="pt").to(device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=400, do_sample=False, num_beams=1)
    return processor.tokenizer.decode(out[0, inputs["input_ids"].shape[-1]:],
                                      skip_special_tokens=True).strip()

## Metrics (WER, CER, Error Counts)

In [5]:
import jiwer
import re
import string

def normalize(text):
    text = text.translate(str.maketrans('', '', string.punctuation)).lower().strip()
    text = re.sub(r" +", " ", text).strip()
    return text

## Run Inference on the Test Set

In [14]:
from tqdm.auto import tqdm
import pandas as pd

eval_rows = []
for sample in tqdm(dataset[:100]):
    model_output = transcribe(torchaudio.load(sample["audio"], normalize=True)[0])
    wer = jiwer.wer(sample["text"], model_output)
    cer = jiwer.cer(sample["text"], model_output)
    
    eval_rows.append({"audio_path": sample["audio"],
                      "label": normalize(sample["text"]),
                      "prediction": normalize(model_output),
                      "wer" : wer,
                      "cer": cer
                     }
                 )

results = pd.DataFrame(eval_rows)
results.head()

  0%|          | 0/100 [00:00<?, ?it/s]

,audio_path,label,prediction,wer,cer
0,/training-1/asr/voxpopuli/out/transcribed_data...,then we as a parliament could take our respons...,we as a parliament could take our responsibili...,0.160000,0.143836
1,/training-1/asr/voxpopuli/out/transcribed_data...,therefore we would like to protest against cha...,therefore we would like to protest against cha...,0.090909,0.029851
2,/training-1/asr/voxpopuli/out/transcribed_data...,we need to take a quick decision if we want to...,we need to take a quick decision if we want to...,0.156250,0.076923
3,/training-1/asr/voxpopuli/out/transcribed_data...,it seemed that this was the end of the cold wa...,it seemed that this was the end of the cold wa...,0.000000,0.000000
4,/training-1/asr/voxpopuli/out/transcribed_data...,the cost of hiv tb and hcv medicines for treat...,the cost of hiv tb and hcv medicines for treat...,0.000000,0.000000


## Corpus-Level Results

In [15]:
print("Mean WER:", results["wer"].mean())
print("Mean CER:", results["cer"].mean())
results[["wer", "cer"]].describe()

Mean WER: 0.056960487886768385
Mean CER: 0.032182195522345775


,wer,cer
count,100.000000,100.000000
mean,0.056960,0.032182
std,0.066200,0.042845
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.040833,0.015969
75%,0.090909,0.046215
max,0.312500,0.206897


## Worst Samples

In [16]:
results.sort_values("wer", ascending=False).head(10)


,audio_path,label,prediction,wer,cer
54,/training-1/asr/voxpopuli/out/transcribed_data...,is there contact with the syrian national coun...,are there contacts with the syrian national co...,0.312500,0.206897
35,/training-1/asr/voxpopuli/out/transcribed_data...,it is unacceptable and as has been said a sign...,i think it is unacceptable and it has been sai...,0.250000,0.127907
78,/training-1/asr/voxpopuli/out/transcribed_data...,should these be offensive then they can be tac...,and should these be offensive then they can be...,0.230769,0.095890
85,/training-1/asr/voxpopuli/out/transcribed_data...,good wording on edis which we believe are very...,good wording on edis which we believe is very ...,0.200000,0.071429
17,/training-1/asr/voxpopuli/out/transcribed_data...,however one aspect was left out the situation ...,one aspect was left aside the situation of tho...,0.186047,0.150198
19,/training-1/asr/voxpopuli/out/transcribed_data...,the situation in which social and economic ris...,the situation in which the social and economic...,0.170213,0.066667
53,/training-1/asr/voxpopuli/out/transcribed_data...,i would like to hear from mrs ashton what kind...,i would like to hear from lady ashton what kin...,0.166667,0.083333
0,/training-1/asr/voxpopuli/out/transcribed_data...,then we as a parliament could take our respons...,we as a parliament could take our responsibili...,0.160000,0.143836
2,/training-1/asr/voxpopuli/out/transcribed_data...,we need to take a quick decision if we want to...,we need to take a quick decision if we want to...,0.156250,0.076923
59,/training-1/asr/voxpopuli/out/transcribed_data...,she accepted the fact that citizenship is subj...,she accepted the fact that citizenship is a na...,0.156250,0.106145


In [18]:
from IPython.display import Audio, display

for _, row in results.sort_values("wer", ascending=False).head(10).iterrows():
    print("=" * 70)
    print("AUDIO  :", row["audio_path"])
    print("LABEL    :", row["label"])
    print("PRED    :", row["prediction"])
    print(f"WER {row["wer"]:.2%} | CER {row["cer"]:.2%}")
    display(Audio(torchaudio.load(row["audio_path"], normalize=True)[0], rate=16000))


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2012/20120612-0900-PLENARY-11-en_20120612-17:59:38_5.wav
LABEL    : is there contact with the syrian national council and if so could she elaborate on this
PRED    : are there contacts with the syrian national council and if so could you elaborate on these contacts
WER 31.25% | CER 20.69%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2018/20180312-0900-PLENARY-14-en_20180312-20:14:51_4.wav
LABEL    : it is unacceptable and as has been said a sign of general regression for womens rights
PRED    : i think it is unacceptable and it has been said a sign of general regression for womens rights
WER 25.00% | CER 12.79%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2017/20170405-0900-PLENARY-13-en_20170405-15:58:20_4.wav
LABEL    : should these be offensive then they can be tackled by a forthright debate
PRED    : and should these be offensive then they can be tackled via forthright debate
WER 23.08% | CER 9.59%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2018/20180301-0900-PLENARY-3-en_20180301-08:47:32_5.wav
LABEL    : good wording on edis which we believe are very important
PRED    : good wording on edis which we believe is very important
WER 20.00% | CER 7.14%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2009/20090310-0900-PLENARY-20-en_20090310-22:15:43_4.wav
LABEL    : however one aspect was left out the situation of thousands of european children left behind by parents who go to work in another european country generally called the migration orphans of which there are almost three hundred and fifty zero in my country
PRED    : one aspect was left aside the situation of thousands of european children left behind by their parents who go to work in another european country generally called the migration orphans almost three hundred and fifty zero in my country
WER 18.60% | CER 15.02%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2009/20090310-0900-PLENARY-20-en_20090310-22:11:44_2.wav
LABEL    : the situation in which social and economic risks are made worse by the absence of parents who cannot meet their childrens needs for care protection and education may lead to an increase in vulnerability the primary responsibility for the development of the child rests with the parents
PRED    : the situation in which the social and economic risks are made worse by the absence of parents who cannot meet their childrens need for care protection and education may bring about increased vulnerability the primary responsibility for the development of the child rests with the parents
WER 17.02% | CER 6.67%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2012/20120612-0900-PLENARY-11-en_20120612-17:59:38_4.wav
LABEL    : i would like to hear from mrs ashton what kind of contact there is with syrian opposition groups
PRED    : i would like to hear from lady ashton what kind of contacts there are with syrian opposition groups
WER 16.67% | CER 8.33%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2013/20131022-0900-PLENARY-3-en_20131022-08:32:08_10.wav
LABEL    : then we as a parliament could take our responsibility and quickly vote through these measures because they will need to take effect on one january
PRED    : we as a parliament could take our responsibility and quickly vote through these measures because they need to take effect actually only one january
WER 16.00% | CER 14.38%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2013/20131022-0900-PLENARY-3-en_20131022-08:32:08_9.wav
LABEL    : we need to take a quick decision if we want to follow the normal procedure i would urge the council to withdraw its legal basis and to accept the commissions legal basis
PRED    : we need to take a quick decision if we want to follow the normal procedure i would urge the council to withdraw their legal base and accept the commission legal base
WER 15.62% | CER 7.69%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2014/20140115-0900-PLENARY-15-en_20140115-18:48:29_5.wav
LABEL    : she accepted the fact that citizenship is subject to national jurisdiction but she also said that according to the maastricht treaty and she is right there has to be a direct link
PRED    : she accepted the fact that citizenship is a national part of the national jurisdiction but she also said that according to the maastricht treaty and she is right there has to be a direct link
WER 15.62% | CER 10.61%
